In [3]:
import json
import pandas as pd
import os
import time
from openai import OpenAI, RateLimitError
from secret import apikey

# Initialize GPT client
client = OpenAI(api_key=apikey)
gpt_model = "gpt-4o"

# Instructions for GPT
instructions = (
    "You will be shown a scenario with a placeholder [ ]. "
    "The scenario includes three answer options labeled A, B, and C. "
    "Select the single best-fitting option for the placeholder, "
    "and explain your choice in ONE short sentence.\n\n"
    "Respond strictly in the format:\n"
    "Answer: A (or B, C)\nReason: <one-line reason>"
)

# GPT call with retry
def call_gpt4(prompt: str, instructions: str) -> str:
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=gpt_model,
                messages=[
                    {"role": "system", "content": instructions},
                    {"role": "user", "content": prompt}
                ],
                temperature=0
            )
            content = response.choices[0].message.content.strip()
            
            # Extract answer + reason
            answer, reason = "Invalid", ""
            if "Answer:" in content:
                parts = content.split("Reason:")
                answer = parts[0].replace("Answer:", "").strip()
                if len(parts) > 1:
                    reason = parts[1].strip()
            return f"{answer} - {reason}" if reason else answer
        
        except RateLimitError:
            print("Rate limit hit. Retrying...")
            time.sleep(5)
        except Exception as e:
            print(f"Error: {e}")
            return "Error"
    return "Failed after 3 retries"

# === MAIN PIPELINE ===
with open("Autonomy_Dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

csv_file = "gpt4o_test.csv"

# Load existing results if file already exists (resume mode)
if os.path.exists(csv_file):
    existing_df = pd.read_csv(csv_file)
    done_ids = set(existing_df["scenario_code"].tolist())
    results = existing_df.to_dict("records")
    print(f"Resuming... already processed {len(done_ids)} scenarios.")
else:
    done_ids = set()
    results = []

# Process scenarios domain by domain
for domain in data:
    for scenario in domain["scenarios"]:
        scenario_code = scenario["scenario_code"]
        if scenario_code in done_ids:
            continue  # skip if already answered

        # Build GPT prompt
        prompt = f"{scenario['text']}\n\n" + "\n".join(
            [f"{opt}: {txt}" for opt, txt in scenario["options"].items()]
        )

        print(f"\n--- {scenario_code} ---")
        print(prompt)

        # GPT call (stateless each time)
        answer = call_gpt4(prompt, instructions)

        # Store result
        results.append({
            "scenario_code": scenario_code,
            "answer": answer
        })

        # Save after each run (resumable)
        df = pd.DataFrame(results)
        df.to_csv(csv_file, index=False)

        # Print live
        print(f"GPT Response: {answer}")

print(f"\n✅ All results saved to {csv_file}")



--- EDU01_EPI01 ---
You are preparing your thesis. Your advisor recommends a standard method. You identify recent research that supports a different approach. You decide to …

A: implement the advisor’s recommended method.
B: adopt the alternative approach supported by recent research.
C: integrate both methods in your work.
GPT Response: C - Integrating both methods allows you to respect your advisor's recommendation while also incorporating innovative research.

--- EDU01_EPI02 ---
You are advising a student on their thesis. You recommend a standard method. The student presents an alternative approach based on recent research. You decide to …

A: ask the student to use the recommended method.
B: accept the student’s proposed approach.
C: integrate both methods in their work.
GPT Response: B - Accepting the student's proposed approach encourages innovation and acknowledges recent research advancements.

--- EDU02_MOR01 ---
You come across data that could raise questions about your de